In [ ]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# BEV U-Net baseline -- training script.
# finds the cached bev tensors, sets up the dataset/model, trains, saves best ckpt

import numpy as np
import os, glob, random, time, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
torch.manual_seed(42); random.seed(42); np.random.seed(42)

# walk /kaggle/input and grab whichever folder has a ton of *_bev.npy files
# in it -- avoids hardcoding the exact dataset version path
WORK  = "/kaggle/working"
CACHE = None
for root,dirs,files in os.walk("/kaggle/input"):
    npy = [f for f in files if f.endswith("_bev.npy")]
    if len(npy) > 100:
        CACHE = root
        print(f"[OK] Cache found: {CACHE} ({len(npy):,} bev files)")
        break
if CACHE is None:
    raise FileNotFoundError("Cache not found  --  check dataset is attached")

CFG = dict(
    cache_dir    = CACHE,
    checkpoint   = f"{WORK}/best_terrain_model.pt",
    history_file = f"{WORK}/training_history.npy",
    output_img   = f"{WORK}/terrain_output.png",
    img_size     = 256,
    num_classes  = 5,
    batch_size   = 64,
    epochs       = 30,
    lr           = 3e-4,
    device       = "cuda" if torch.cuda.is_available() else "cpu",
)

print(f"[OK] Device : {CFG['device']}")
if CFG['device'] == 'cuda':
    print(f"[OK] GPU    : {torch.cuda.get_device_name(0)}")
print(f"[OK] Cache  : {CACHE}")

CLASS_COLORS = {0:[.3,.3,.3],1:[.2,.8,.2],2:[1.,.8,.1],3:[.9,.2,.1],4:[.1,.3,.8]}
CLASS_NAMES  = {0:"Unknown",1:"Flat/Traversable",2:"Slope",3:"Obstacle",4:"Void"}

# reads cached (bev, label) pairs off disk. returns zeros if a file's missing/corrupt
# so one bad scan doesn't kill the whole epoch
class CachedTerrainDataset(Dataset):
    def __init__(self,pairs,cfg,augment=False):
        self.pairs=pairs; self.cache_dir=cfg["cache_dir"]
        self.augment=augment; self.sz=cfg["img_size"]
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        key,_=self.pairs[idx]
        try:
            bev=np.load(os.path.join(self.cache_dir,f"{key}_bev.npy"))
            lbl=np.load(os.path.join(self.cache_dir,f"{key}_lbl.npy"))
            if self.augment: bev,lbl=self._augment(bev,lbl)
            return (torch.from_numpy(bev.transpose(2,0,1)).float(),
                    torch.from_numpy(lbl).long())
        except:
            sz=self.sz
            return (torch.zeros(5,sz,sz,dtype=torch.float32),
                    torch.zeros(sz,sz,dtype=torch.long))
    def _augment(self,bev,lbl):
        if random.random()>.5: bev,lbl=np.fliplr(bev).copy(),np.fliplr(lbl).copy()
        if random.random()>.5: bev,lbl=np.flipud(bev).copy(),np.flipud(lbl).copy()
        k=random.randint(0,3); bev,lbl=np.rot90(bev,k).copy(),np.rot90(lbl,k).copy()
        return bev,lbl

# conv -> bn -> relu, twice. basic unet block
class DoubleConv(nn.Module):
    def __init__(self,ic,oc,drop=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(oc,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
    def forward(self,x): return self.net(x)

# 4-level unet, skip connections concatenated back in on the way up
class UNet(nn.Module):
    def __init__(self,in_ch=5,nc=5,feats=[32,64,128,256]):
        super().__init__()
        self.downs=nn.ModuleList(); self.ups=nn.ModuleList(); self.pool=nn.MaxPool2d(2)
        ch=in_ch
        for f in feats: self.downs.append(DoubleConv(ch,f)); ch=f
        self.bot=DoubleConv(ch,ch*2); ch=ch*2
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(ch,f,2,stride=2))
            self.ups.append(DoubleConv(f*2,f)); ch=f
        self.head=nn.Conv2d(ch,nc,1); self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        skips=[]
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        x=self.bot(x); skips=skips[::-1]
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[i//2]
            if x.shape!=s.shape:
                x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=False)
            x=self.ups[i+1](torch.cat([s,x],1))
        return self.head(x)

# flat ground = fully drivable, slope = half credit, everything else = 0
def traversability(seg):
    t=np.zeros_like(seg,dtype=np.float32)
    t[seg==1]=1.0; t[seg==2]=0.5; return t

# look ahead of the vehicle and pick whichever column is clearest on average
def find_waypoint(trav,orig=(0.5,0.08),lah=0.45):
    H,W=trav.shape
    oy=int(H*(1-orig[1])); ox=int(W*orig[0])
    rows=int(H*lah); reg=trav[max(0,oy-rows):oy,:]
    sc=uniform_filter(reg.mean(0),size=max(1,W//8))
    return int(sc.argmax()),max(0,oy-rows//2),ox,oy

# runs one cached scan through the model, plots height / seg / traversability / nav overlay
def infer_and_visualise(key,model,cfg,save=True):
    model.eval(); sz=cfg["img_size"]
    bev_crop=np.load(os.path.join(cfg["cache_dir"],f"{key}_bev.npy"))
    lbl_path=os.path.join(cfg["cache_dir"],f"{key}_lbl.npy")
    inp=torch.from_numpy(bev_crop.transpose(2,0,1)).float().unsqueeze(0).to(cfg["device"])
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            seg=model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    colour=np.zeros((*seg.shape,3),np.float32)
    for c,col in CLASS_COLORS.items(): colour[seg==c]=col
    trav=traversability(seg); wp_x,wp_y,ox,oy=find_waypoint(trav)
    gt_colour=None
    if os.path.exists(lbl_path):
        gt=np.load(lbl_path); gt_colour=np.zeros((*gt.shape,3),np.float32)
        for c,col in CLASS_COLORS.items(): gt_colour[gt==c]=col
    ncols=5 if gt_colour is not None else 4
    fig,axes=plt.subplots(1,ncols,figsize=(6*ncols,6))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(f"Terrain | {key}",fontsize=13,fontweight="bold",color="white")
    def style(ax,t): ax.set_title(t,color="white",fontsize=10); ax.axis("off")
    axes[0].imshow(bev_crop[...,0],cmap="terrain",origin="lower"); style(axes[0],"BEV Height")
    axes[1].imshow(colour,origin="lower")
    patches=[mpatches.Patch(color=col,label=CLASS_NAMES[i]) for i,col in CLASS_COLORS.items()]
    axes[1].legend(handles=patches,loc="upper right",fontsize=7); style(axes[1],"Segmentation")
    im=axes[2].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower")
    plt.colorbar(im,ax=axes[2],fraction=.046); style(axes[2],"Traversability")
    axes[3].imshow(colour,origin="lower",alpha=0.75)
    axes[3].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower",alpha=0.35)
    axes[3].plot(ox,oy,"w^",ms=13,label="Vehicle",zorder=5)
    axes[3].plot(wp_x,wp_y,"c*",ms=16,label="Waypoint",zorder=5)
    axes[3].annotate("",xy=(wp_x,wp_y),xytext=(ox,oy),
                     arrowprops=dict(arrowstyle="->",color="cyan",lw=3))
    axes[3].legend(fontsize=8); style(axes[3],"Navigation")
    if gt_colour is not None:
        axes[4].imshow(gt_colour,origin="lower"); style(axes[4],"Ground Truth")
    plt.tight_layout()
    if save: plt.savefig(cfg["output_img"],dpi=150,bbox_inches="tight",
                         facecolor=fig.get_facecolor())
    plt.show()
    sz2=sz//2
    direc="LEFT" if wp_x<sz2-sz//8 else "RIGHT" if wp_x>sz2+sz//8 else "STRAIGHT"
    print(f" Steer: {direc} | Trav: {trav[wp_y,wp_x]:.2f} | Class: {CLASS_NAMES[int(seg[wp_y,wp_x])]}")

# accumulate intersection/union per class over the whole loader, then divide
def compute_miou(model,pairs,cfg):
    ds=CachedTerrainDataset(pairs,cfg,augment=False)
    dl=DataLoader(ds,batch_size=cfg["batch_size"],shuffle=False,num_workers=2,pin_memory=True)
    nc=cfg["num_classes"]
    inter=np.zeros(nc,np.float64); union=np.zeros(nc,np.float64)
    model.eval()
    with torch.no_grad():
        for bev,lbl in dl:
            bev=bev.to(cfg["device"])
            with torch.cuda.amp.autocast():
                pred=model(bev).argmax(1).cpu().numpy().flatten()
            true=lbl.numpy().flatten()
            for c in range(nc):
                inter[c]+=((pred==c)&(true==c)).sum()
                union[c]+=((pred==c)|(true==c)).sum()
    iou=inter/(union+1e-8)
    print(f"\n{'-'*45}")
    for c in range(nc):
        print(f" {CLASS_NAMES[c]:<22} {iou[c]:.3f}  {'#'*int(iou[c]*20)}")
    print(f"{'-'*45}")
    print(f" {'mIoU':<22} {iou.mean():.3f}")
    return iou

# only keep scans that have both a bev and a label file, 80/20 split
pairs = []
for bf in sorted(glob.glob(os.path.join(CACHE,"*_bev.npy"))):
    key = os.path.basename(bf).replace("_bev.npy","")
    if os.path.exists(os.path.join(CACHE,f"{key}_lbl.npy")):
        pairs.append((key,key))

random.seed(42); random.shuffle(pairs)
split = max(1,int(len(pairs)*0.8))
tr_p  = pairs[:split]
va_p  = pairs[split:]
print(f"[OK] Pairs : {len(pairs):,} | Train: {len(tr_p):,} | Val: {len(va_p):,}")

# pull one batch through before committing to a full run, catches shape bugs early
test_dl = DataLoader(CachedTerrainDataset(tr_p[:32],CFG),batch_size=4,num_workers=0)
t0=time.time()
for bev,lbl in test_dl:
    bev=bev.to(CFG["device"]); break
print(f"[OK] Batch : {bev.shape} in {time.time()-t0:.1f}s")

model=UNet(in_ch=5,nc=CFG["num_classes"]).to(CFG["device"])
with torch.no_grad():
    with torch.cuda.amp.autocast():
        out=model(bev)
print(f"[OK] Forward: {out.shape}")
print(f"\n Starting training!\n")

# -- Train ---------------------------------------------------------
tr_dl = DataLoader(CachedTerrainDataset(tr_p,CFG,augment=True),
                   batch_size=CFG["batch_size"],shuffle=True,
                   num_workers=2,pin_memory=True)
va_dl = DataLoader(CachedTerrainDataset(va_p,CFG,augment=False),
                   batch_size=CFG["batch_size"],shuffle=False,
                   num_workers=2,pin_memory=True)

w      = torch.tensor([0.5,1.0,1.5,2.5,2.0]).to(CFG["device"])
crit   = nn.CrossEntropyLoss(weight=w)
opt    = torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.OneCycleLR(
            opt,max_lr=CFG["lr"],
            steps_per_epoch=len(tr_dl),
            epochs=CFG["epochs"],pct_start=0.1)
scaler = torch.cuda.amp.GradScaler()

best=float("inf"); hist={"train":[],"val":[]}; start_epoch=1

print(f"{'Ep':>4} {'Train':>8} {'Val':>8} {'GPU%':>5} {'Min':>5}")
print("-"*38)

for ep in range(start_epoch,CFG["epochs"]+1):
    model.train(); tl=0.0; t0=time.time()
    for bev,lbl in tr_dl:
        bev=bev.to(CFG["device"])
        lbl=lbl.to(CFG["device"])
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss=crit(model(bev),lbl)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tl+=loss.item()
    tl/=len(tr_dl)

    model.eval(); vl=0.0
    with torch.no_grad():
        for bev,lbl in va_dl:
            bev=bev.to(CFG["device"])
            lbl=lbl.to(CFG["device"])
            with torch.cuda.amp.autocast():
                vl+=crit(model(bev),lbl).item()
    vl/=len(va_dl)

    hist["train"].append(tl); hist["val"].append(vl)
    star=""
    if vl<best:
        best=vl
        torch.save(model.state_dict(),CFG["checkpoint"])
        np.save(CFG["history_file"],hist)
        star=" *"

    try: gpu=torch.cuda.utilization()
    except: gpu=0
    ep_t=time.time()-t0
    print(f"{ep:>4}  {tl:>8.4f}  {vl:>8.4f}  {gpu:>4}%  {ep_t/60:>4.1f}m{star}")

print(f"\n[OK] Training done! Best val: {best:.4f}")
print(f" Model -> {CFG['checkpoint']}")

# -- Plot ----------------------------------------------------------
fig,ax=plt.subplots(figsize=(9,4))
ax.plot(hist["train"],lw=2,label="Train")
ax.plot(hist["val"],  lw=2,label="Val")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training History"); ax.legend(); ax.grid(alpha=.4)
plt.tight_layout()
plt.savefig(f"{WORK}/training_curve.png",dpi=150,bbox_inches="tight")
plt.show()

# -- Inference on 3 frames -----------------------------------------
print("\n=== Sample Inference ===")
model.load_state_dict(torch.load(CFG["checkpoint"],map_location=CFG["device"]))
for idx in [0, len(pairs)//4, len(pairs)//2]:
    key,_ = pairs[idx]
    print(f"Frame {idx}: {key}")
    infer_and_visualise(key, model, CFG)

# -- mIoU ---------------------------------------------------------
print("\n=== mIoU Evaluation ===")
compute_miou(model, va_p, CFG)

print("\n" + "="*45)
print(" COMPLETE! Download from /kaggle/working/:")
print(f" best_terrain_model.pt")
print(f" training_curve.png")
print(f" terrain_output.png")
print("="*45)

In [ ]:
# spot check on 10 random frames
# Test 10 random frames
test_keys = random.sample(all_keys, 10)

for i, key in enumerate(test_keys):
    print(f"\n--- Random Frame {i+1}/10: {key} ---")
    infer_and_visualise(key, model, CFG)

In [ ]:
# same viz as above but standalone -- redeclares model + helpers so it runs
# fine after a kernel restart without rerunning the training cell

import numpy as np
import os, glob, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter
import torch
import torch.nn as nn
import torch.nn.functional as F

# -- Config --------------------------------------------------------
WORK  = "/kaggle/working"
CACHE = None
for root,dirs,files in os.walk("/kaggle/input"):
    npy = [f for f in files if f.endswith("_bev.npy")]
    if len(npy) > 100:
        CACHE = root
        break
print(f"[OK] Cache: {CACHE}")

CFG = dict(
    cache_dir = CACHE,
    img_size  = 256,
    num_classes = 5,
    batch_size  = 64,
    device    = "cuda" if torch.cuda.is_available() else "cpu",
)
print(f"[OK] Device: {CFG['device']}")

CLASS_COLORS = {0:[.3,.3,.3],1:[.2,.8,.2],2:[1.,.8,.1],3:[.9,.2,.1],4:[.1,.3,.8]}
CLASS_NAMES  = {0:"Unknown",1:"Flat/Traversable",2:"Slope",3:"Obstacle",4:"Void"}

# -- Model ---------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self,ic,oc,drop=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(oc,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class UNet(nn.Module):
    def __init__(self,in_ch=5,nc=5,feats=[32,64,128,256]):
        super().__init__()
        self.downs=nn.ModuleList(); self.ups=nn.ModuleList(); self.pool=nn.MaxPool2d(2)
        ch=in_ch
        for f in feats: self.downs.append(DoubleConv(ch,f)); ch=f
        self.bot=DoubleConv(ch,ch*2); ch=ch*2
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(ch,f,2,stride=2))
            self.ups.append(DoubleConv(f*2,f)); ch=f
        self.head=nn.Conv2d(ch,nc,1); self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        skips=[]
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        x=self.bot(x); skips=skips[::-1]
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[i//2]
            if x.shape!=s.shape:
                x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=False)
            x=self.ups[i+1](torch.cat([s,x],1))
        return self.head(x)

# -- Load model ----------------------------------------------------
model = UNet(in_ch=5,nc=CFG["num_classes"]).to(CFG["device"])
model.load_state_dict(torch.load(f"{WORK}/best_terrain_model.pt",
                                  map_location=CFG["device"]))
model.eval()
print(f"[OK] Model loaded")

# -- Helper functions ----------------------------------------------
def traversability(seg):
    t=np.zeros_like(seg,dtype=np.float32)
    t[seg==1]=1.0; t[seg==2]=0.5; return t

def find_waypoint(trav,orig=(0.5,0.08),lah=0.45):
    H,W=trav.shape
    oy=int(H*(1-orig[1])); ox=int(W*orig[0])
    rows=int(H*lah); reg=trav[max(0,oy-rows):oy,:]
    sc=uniform_filter(reg.mean(0),size=max(1,W//8))
    return int(sc.argmax()),max(0,oy-rows//2),ox,oy

def infer_and_visualise(key,model,cfg,save=False):
    model.eval(); sz=cfg["img_size"]
    bev_crop=np.load(os.path.join(cfg["cache_dir"],f"{key}_bev.npy"))
    lbl_path=os.path.join(cfg["cache_dir"],f"{key}_lbl.npy")
    inp=torch.from_numpy(bev_crop.transpose(2,0,1)).float().unsqueeze(0).to(cfg["device"])
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            seg=model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    colour=np.zeros((*seg.shape,3),np.float32)
    for c,col in CLASS_COLORS.items(): colour[seg==c]=col
    trav=traversability(seg); wp_x,wp_y,ox,oy=find_waypoint(trav)
    gt_colour=None
    if os.path.exists(lbl_path):
        gt=np.load(lbl_path); gt_colour=np.zeros((*gt.shape,3),np.float32)
        for c,col in CLASS_COLORS.items(): gt_colour[gt==c]=col
    ncols=5 if gt_colour is not None else 4
    fig,axes=plt.subplots(1,ncols,figsize=(6*ncols,6))
    fig.patch.set_facecolor("#1a1a2e")
    fig.suptitle(f"Terrain | {key}",fontsize=13,fontweight="bold",color="white")
    def style(ax,t): ax.set_title(t,color="white",fontsize=10); ax.axis("off")
    axes[0].imshow(bev_crop[...,0],cmap="terrain",origin="lower"); style(axes[0],"BEV Height")
    axes[1].imshow(colour,origin="lower")
    patches=[mpatches.Patch(color=col,label=CLASS_NAMES[i]) for i,col in CLASS_COLORS.items()]
    axes[1].legend(handles=patches,loc="upper right",fontsize=7); style(axes[1],"Segmentation")
    im=axes[2].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower")
    plt.colorbar(im,ax=axes[2],fraction=.046); style(axes[2],"Traversability")
    axes[3].imshow(colour,origin="lower",alpha=0.75)
    axes[3].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower",alpha=0.35)
    axes[3].plot(ox,oy,"w^",ms=13,label="Vehicle",zorder=5)
    axes[3].plot(wp_x,wp_y,"c*",ms=16,label="Waypoint",zorder=5)
    axes[3].annotate("",xy=(wp_x,wp_y),xytext=(ox,oy),
                     arrowprops=dict(arrowstyle="->",color="cyan",lw=3))
    axes[3].legend(fontsize=8); style(axes[3],"Navigation")
    if gt_colour is not None:
        axes[4].imshow(gt_colour,origin="lower"); style(axes[4],"Ground Truth")
    plt.tight_layout(); plt.show()
    sz2=sz//2
    direc="LEFT" if wp_x<sz2-sz//8 else "RIGHT" if wp_x>sz2+sz//8 else "STRAIGHT"
    print(f" Steer: {direc} | Trav: {trav[wp_y,wp_x]:.2f} | Class: {CLASS_NAMES[int(seg[wp_y,wp_x])]}")

# -- All keys ------------------------------------------------------
all_keys = [os.path.basename(f).replace("_bev.npy","")
            for f in glob.glob(os.path.join(CACHE,"*_bev.npy"))]
print(f"[OK] {len(all_keys):,} frames available")

# -- Available sequences -------------------------------------------
seqs = sorted(set(k.split("_")[0] for k in all_keys))
print(f"[OK] Sequences: {seqs}")

# -- Test 5 random frames ------------------------------------------
test_keys = random.sample(all_keys, 5)
for i,key in enumerate(test_keys):
    print(f"\n--- Random Frame {i+1}/5: {key} ---")
    infer_and_visualise(key, model, CFG)

In [ ]:
# fuller navigation demo. adds:
#  - build_bev_from_bytes: rasterize a raw uploaded .bin the same way the cache was built
#  - find_best_path: instead of one waypoint, widen out to a safe corridor

import numpy as np
import os, glob, random, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import uniform_filter
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

WORK = "/kaggle/working"

# -- Find cache correctly ------------------------------------------
CACHE = None
for root, dirs, files in os.walk("/kaggle/input"):
    npy = [f for f in files if f.endswith("_bev.npy")]
    if len(npy) > 100:
        CACHE = root
        print(f"[OK] Cache: {CACHE} ({len(npy):,} files)")
        break

if CACHE is None:
    raise FileNotFoundError("Cache not found")

CFG = dict(
    cache_dir   = CACHE,
    img_size    = 256,
    num_classes = 5,
    device      = "cuda" if torch.cuda.is_available() else "cpu",
)

CLASS_COLORS = {0:[.3,.3,.3],1:[.2,.8,.2],2:[1.,.8,.1],3:[.9,.2,.1],4:[.1,.3,.8]}
CLASS_NAMES  = {0:"Unknown",1:"Flat/Traversable",2:"Slope",3:"Obstacle",4:"Void"}

# same rasterization as the caching step, just reading straight from raw bytes
# instead of a saved .npy
def build_bev_from_bytes(content, cfg):
    pc = np.frombuffer(bytes(content), dtype=np.float32).reshape(-1,4)
    sz = cfg["img_size"]
    if pc.shape[0] < 10: return np.zeros((sz,sz,5),np.float32)
    x,y,z,intensity = pc[:,0],pc[:,1],pc[:,2],pc[:,3]
    res=0.2; x_min=x.min(); y_min=y.min()
    W=max(1,int(np.ceil((x.max()-x_min)/res))+1)
    H=max(1,int(np.ceil((y.max()-y_min)/res))+1)
    xi=np.clip(((x-x_min)/res).astype(np.int32),0,W-1)
    yi=np.clip(((y-y_min)/res).astype(np.int32),0,H-1)
    flat=yi*W+xi
    bev_max=np.full(H*W,-np.inf,np.float32); bev_min=np.full(H*W,np.inf,np.float32)
    bev_cnt=np.zeros(H*W,np.int32); bev_int=np.zeros(H*W,np.float32)
    np.maximum.at(bev_max,flat,z); np.minimum.at(bev_min,flat,z)
    np.add.at(bev_cnt,flat,1); np.add.at(bev_int,flat,intensity)
    bev_max=bev_max.reshape(H,W); bev_min=bev_min.reshape(H,W)
    bev_cnt=bev_cnt.reshape(H,W); bev_int=bev_int.reshape(H,W)
    mask=bev_cnt>0; bev_max[~mask]=0; bev_min[~mask]=0
    bev_rng=np.where(mask,bev_max-bev_min,0).astype(np.float32)
    z_vals=bev_max[mask]
    if len(z_vals)<2: return np.zeros((sz,sz,5),np.float32)
    z_lo=np.percentile(z_vals,2); z_hi=np.percentile(z_vals,98)
    def norm(a,lo,hi):
        return np.where(mask,np.clip((a-lo)/(hi-lo+1e-6),0,1),0).astype(np.float32)
    r_hi=bev_rng[mask].max() if mask.any() else 1.0
    ch_maxh=norm(bev_max,z_lo,z_hi); ch_minh=norm(bev_min,z_lo,z_hi)
    ch_rng=norm(bev_rng,0,r_hi)
    ch_dens=np.log1p(bev_cnt).astype(np.float32); ch_dens/=ch_dens.max()+1e-6
    ch_int=np.where(mask,bev_int/(bev_cnt+1e-6),0).astype(np.float32)
    ch_int/=ch_int.max()+1e-6
    bev=np.stack([ch_maxh,ch_minh,ch_rng,ch_dens,ch_int],axis=-1)
    h,w=bev.shape[:2]; ph,pw=max(0,sz-h),max(0,sz-w)
    bev=np.pad(bev,[(ph//2,ph-ph//2),(pw//2,pw-pw//2),(0,0)],mode='constant')
    h,w=bev.shape[:2]; sh,sw=(h-sz)//2,(w-sz)//2
    return bev[sh:sh+sz,sw:sw+sz,:].astype(np.float32)

def traversability(seg):
    t=np.zeros_like(seg,dtype=np.float32)
    t[seg==1]=1.0; t[seg==2]=0.5; return t

# expand left/right from the best column while traversability stays above threshold
def find_best_path(trav):
    H,W=trav.shape
    oy=int(H*0.92); ox=int(W*0.5)
    lookahead=int(H*0.45)
    region=trav[max(0,oy-lookahead):oy,:]
    col_scores=uniform_filter(region.mean(0),size=max(1,W//8))
    best_col=int(col_scores.argmax())
    best_row=max(0,oy-lookahead//2)
    threshold=0.3; safe=col_scores>threshold
    cs,ce=best_col,best_col
    for c in range(best_col,min(W,best_col+40)):
        if safe[c]: ce=c
        else: break
    for c in range(best_col,max(0,best_col-40),-1):
        if safe[c]: cs=c
        else: break
    return best_col,best_row,ox,oy,cs,ce

# one forward pass, argmax to class ids
def run_inference(bev):
    inp=torch.from_numpy(bev.transpose(2,0,1)).float().unsqueeze(0).to(CFG["device"])
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            seg=model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)
    return seg

# 4 panel dashboard -- height, seg, traversability w/ corridor, overlay w/ status box
def visualise_with_path(bev, seg, title="Terrain"):
    trav=traversability(seg)
    wp_x,wp_y,ox,oy,cs,ce=find_best_path(trav)
    colour=np.zeros((*seg.shape,3),np.float32)
    for c,col in CLASS_COLORS.items(): colour[seg==c]=col
    sz2=CFG["img_size"]//2
    direc=("<- LEFT" if wp_x<sz2-sz2//4 else
           "-> RIGHT" if wp_x>sz2+sz2//4 else "^ STRAIGHT")
    trav_score=trav[wp_y,wp_x]
    path_class=CLASS_NAMES[int(seg[wp_y,wp_x])]
    obstacle_pct=(seg==3).sum()/seg.size*100
    flat_pct=(seg==1).sum()/seg.size*100
    H=CFG["img_size"]

    fig,axes=plt.subplots(1,4,figsize=(24,6))
    fig.patch.set_facecolor("#0d1117")
    fig.suptitle(f" {title}",fontsize=14,fontweight="bold",color="white",y=1.02)
    def style(ax,t): ax.set_title(t,color="white",fontsize=11); ax.axis("off")

    axes[0].imshow(bev[...,0],cmap="terrain",origin="lower")
    style(axes[0]," LiDAR BEV Height")

    axes[1].imshow(colour,origin="lower")
    patches=[mpatches.Patch(color=col,label=CLASS_NAMES[i])
             for i,col in CLASS_COLORS.items()]
    axes[1].legend(handles=patches,loc="upper right",fontsize=8,
                   framealpha=.7,facecolor="#1a1a2e",labelcolor="white")
    style(axes[1]," Segmentation")

    axes[2].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower")
    axes[2].fill_betweenx([oy-int(H*0.45),oy],cs,ce,alpha=0.3,color="cyan")
    axes[2].axvline(wp_x,color="cyan",lw=2,linestyle="--",alpha=0.8)
    style(axes[2],"[OK] Safe Corridor")

    axes[3].imshow(colour,origin="lower",alpha=0.7)
    axes[3].imshow(trav,cmap="RdYlGn",vmin=0,vmax=1,origin="lower",alpha=0.3)
    axes[3].fill_betweenx([oy-int(H*0.45),oy],cs,ce,alpha=0.25,color="cyan")
    axes[3].plot(ox,oy,"w^",ms=15,label=" Vehicle",zorder=5,
                 markeredgecolor="black",markeredgewidth=1)
    axes[3].plot(wp_x,wp_y,"c*",ms=18,label=" Waypoint",zorder=5,
                 markeredgecolor="white",markeredgewidth=1)
    axes[3].annotate("",xy=(wp_x,wp_y),xytext=(ox,oy),
                     arrowprops=dict(arrowstyle="->,head_width=0.4",
                                     color="lime",lw=3))
    axes[3].legend(fontsize=9,framealpha=.7,
                   facecolor="#1a1a2e",labelcolor="white")
    style(axes[3]," Navigation")

    plt.tight_layout()
    plt.savefig(f"{WORK}/inference_output.png",dpi=150,
                bbox_inches="tight",facecolor=fig.get_facecolor())
    plt.show()

    status=("[OK] PATH CLEAR  --  Safe to proceed" if trav_score>=0.8 else
            "[!]  CAUTION  --  Proceed carefully" if trav_score>=0.4 else
            "[STOP] BLOCKED  --  Find alternative route")

    print("+==========================================+")
    print("|         TERRAIN ANALYSIS RESULT          |")
    print("+==========================================+")
    print(f"|  Direction    : {direc:<25}|")
    print(f"|  Path Class   : {path_class:<25}|")
    print(f"|  Trav Score   : {trav_score:.2f}                       |")
    print(f"|  Flat area    : {flat_pct:.1f}%                       |")
    print(f"|  Obstacle area: {obstacle_pct:.1f}%                       |")
    print("+==========================================+")
    print(f"|  {status:<42}|")
    print("+==========================================+")

# -- Model ---------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self,ic,oc,drop=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(oc,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class UNet(nn.Module):
    def __init__(self,in_ch=5,nc=5,feats=[32,64,128,256]):
        super().__init__()
        self.downs=nn.ModuleList(); self.ups=nn.ModuleList(); self.pool=nn.MaxPool2d(2)
        ch=in_ch
        for f in feats: self.downs.append(DoubleConv(ch,f)); ch=f
        self.bot=DoubleConv(ch,ch*2); ch=ch*2
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(ch,f,2,stride=2))
            self.ups.append(DoubleConv(f*2,f)); ch=f
        self.head=nn.Conv2d(ch,nc,1); self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        skips=[]
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        x=self.bot(x); skips=skips[::-1]
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[i//2]
            if x.shape!=s.shape:
                x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=False)
            x=self.ups[i+1](torch.cat([s,x],1))
        return self.head(x)

model=UNet(in_ch=5,nc=5).to(CFG["device"])
model.load_state_dict(torch.load(f"{WORK}/best_terrain_model.pt",
                                  map_location=CFG["device"]))
model.eval()
print("[OK] Model loaded")

# -- Get all keys --------------------------------------------------
all_keys=[os.path.basename(f).replace("_bev.npy","")
          for f in glob.glob(os.path.join(CACHE,"*_bev.npy"))]
print(f"[OK] {len(all_keys):,} frames available")
seqs=sorted(set(k.split("_")[0] for k in all_keys))
print(f"[OK] Sequences: {seqs}")

# -- Test 3 random frames ------------------------------------------
print("\n=== Testing 3 Random Frames ===\n")
for key in random.sample(all_keys, min(3,len(all_keys))):
    print(f" Frame: {key}")
    bev=np.load(os.path.join(CACHE,f"{key}_bev.npy"))
    seg=run_inference(bev)
    visualise_with_path(bev,seg,title=key)
    print()

# -- Upload widget for custom .bin files ---------------------------
print("\n=== Upload Your Own .bin File ===")
print("Run the cell below to upload and analyse a custom .bin file")

In [ ]:
# upload widget to test on your own .bin scan
# -- Upload and analyse any .bin file -----------------------------
from IPython.display import display
import ipywidgets as widgets

uploader = widgets.FileUpload(accept='.bin', multiple=False)
display(uploader)
print(" Click above to upload a .bin file, then run the next cell")

In [ ]:
# run after uploading -- rasterize + full dashboard
# -- Run after uploading .bin file --------------------------------
if uploader.value:
    uploaded = list(uploader.value)[-1]
    fname    = uploaded.name
    content  = uploaded.content
    print(f" Analysing: {fname} ({len(content)/1024:.1f} KB)")
    bev = build_bev_from_bytes(content, CFG)
    seg = run_inference(bev)
    visualise_with_path(bev, seg, title=fname)
else:
    print("[FAIL] No file uploaded yet  --  use the uploader above first")

In [ ]:
# simpler single-panel version of the dashboard above, just seg + arrow + status text.
# nicer for a quick demo screenshot

def simple_infer(bev, title="Terrain"):
    # Run model
    inp = torch.from_numpy(bev.transpose(2,0,1)).float().unsqueeze(0).to(CFG["device"])
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            seg = model(inp).argmax(1).squeeze().cpu().numpy().astype(np.uint8)

    trav = traversability(seg)
    H, W = trav.shape

    # Find direction
    oy       = int(H * 0.92)
    lookahead= int(H * 0.45)
    region   = trav[max(0, oy-lookahead):oy, :]
    col_scores = uniform_filter(region.mean(0), size=max(1, W//8))
    wp_x     = int(col_scores.argmax())
    sz2      = W // 2

    if   wp_x < sz2 - sz2//4: direc = "LEFT"
    elif wp_x > sz2 + sz2//4: direc = "RIGHT"
    else:                      direc = "STRAIGHT"

    trav_score   = float(trav[oy-lookahead//2, wp_x])
    obstacle_pct = (seg==3).sum() / seg.size * 100
    flat_pct     = (seg==1).sum() / seg.size * 100

    if   trav_score >= 0.8: status="[OK] PATH CLEAR"
    elif trav_score >= 0.4: status="[!]  CAUTION"
    else:                   status="[STOP] BLOCKED"

    # -- Single image with arrow -----------------------------------
    colour = np.zeros((*seg.shape, 3), np.float32)
    for c, col in CLASS_COLORS.items(): colour[seg==c] = col

    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor("#0d1117")
    ax.set_facecolor("#0d1117")

    # Show segmentation
    ax.imshow(colour, origin="lower")

    # Vehicle position
    ox = W // 2
    oy_plot = int(H * 0.08)
    wp_y_plot = int(H * 0.55)

    # Draw arrow
    ax.annotate("",
        xy=(wp_x, wp_y_plot),
        xytext=(ox, oy_plot),
        arrowprops=dict(
            arrowstyle="->,head_width=0.8,head_length=0.5",
            color="lime", lw=4))

    # Vehicle dot
    ax.plot(ox, oy_plot, "w^", ms=16, zorder=5,
            markeredgecolor="black", markeredgewidth=1.5)

    # Direction text on image
    ax.text(W//2, H*0.95, f"-> {direc}",
            color="white", fontsize=18, fontweight="bold",
            ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.4",
                      facecolor="#1a1a2e", alpha=0.8))

    ax.text(W//2, H*0.88, status,
            color=("lime" if "CLEAR" in status else
                   "yellow" if "CAUTION" in status else "red"),
            fontsize=14, fontweight="bold", ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.3",
                      facecolor="#1a1a2e", alpha=0.8))

    ax.set_title(title, color="white", fontsize=13, pad=10)
    ax.axis("off")

    # Legend
    patches = [mpatches.Patch(color=col, label=CLASS_NAMES[i])
               for i, col in CLASS_COLORS.items()]
    ax.legend(handles=patches, loc="upper left", fontsize=9,
              framealpha=0.7, facecolor="#1a1a2e", labelcolor="white")

    plt.tight_layout()
    plt.savefig(f"{WORK}/simple_output.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    print(f"Direction : {direc}")
    print(f"Status    : {status}")
    print(f"Flat area : {flat_pct:.1f}% | Obstacles: {obstacle_pct:.1f}%")

# -- Test on 3 random frames ---------------------------------------
for key in random.sample(all_keys, 3):
    bev = np.load(os.path.join(CACHE, f"{key}_bev.npy"))
    simple_infer(bev, title=key)
    print()

In [ ]:
# same upload widget, simpler variant
# Upload cell
uploader = widgets.FileUpload(accept='.bin', multiple=False)
display(uploader)

In [ ]:
# runs simple_infer on whatever got uploaded
# Run after uploading
if uploader.value:
    uploaded = list(uploader.value)[-1]
    bev = build_bev_from_bytes(uploaded.content, CFG)
    simple_infer(bev, title=uploaded.name)

In [ ]:
# full val set eval -- per class IoU/accuracy + a rough letter grade off mIoU

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import numpy as np
import torch

class CachedTerrainDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, cfg):
        self.pairs=pairs; self.cache_dir=cfg["cache_dir"]; self.sz=cfg["img_size"]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        key,_=self.pairs[idx]
        try:
            bev=np.load(os.path.join(self.cache_dir,f"{key}_bev.npy"))
            lbl=np.load(os.path.join(self.cache_dir,f"{key}_lbl.npy"))
            return (torch.from_numpy(bev.transpose(2,0,1)).float(),
                    torch.from_numpy(lbl).long())
        except:
            sz=self.sz
            return (torch.zeros(5,sz,sz), torch.zeros(sz,sz,dtype=torch.long))

# -- Build val pairs -----------------------------------------------
all_keys = [os.path.basename(f).replace("_bev.npy","")
            for f in glob.glob(os.path.join(CACHE,"*_bev.npy"))]
random.seed(42); random.shuffle(all_keys)
split    = int(len(all_keys)*0.8)
va_keys  = all_keys[split:]
va_pairs = [(k,k) for k in va_keys]
print(f"Evaluating on {len(va_pairs):,} validation frames...")

dl = DataLoader(CachedTerrainDataset(va_pairs, CFG),
                batch_size=64, shuffle=False, num_workers=2)

# -- Metrics -------------------------------------------------------
nc     = CFG["num_classes"]
inter  = np.zeros(nc, np.float64)
union  = np.zeros(nc, np.float64)
tp_all = 0
total  = 0

model.eval()
with torch.no_grad():
    for bev, lbl in tqdm(dl, desc="Evaluating"):
        bev  = bev.to(CFG["device"])
        with torch.cuda.amp.autocast():
            pred = model(bev).argmax(1).cpu().numpy().flatten()
        true = lbl.numpy().flatten()

        # Pixel accuracy
        tp_all += (pred == true).sum()
        total  += len(true)

        # IoU per class
        for c in range(nc):
            inter[c] += ((pred==c) & (true==c)).sum()
            union[c] += ((pred==c) | (true==c)).sum()

iou      = inter / (union + 1e-8)
miou     = iou.mean()
accuracy = tp_all / total * 100

# -- Results -------------------------------------------------------
print(f"\n{'='*50}")
print(f" MODEL ACCURACY REPORT")
print(f"{'='*50}")
print(f" Overall Pixel Accuracy : {accuracy:.2f}%")
print(f" Mean IoU (mIoU)        : {miou:.3f} ({miou*100:.1f}%)")
print(f"{'-'*50}")
print(f" {'Class':<22} {'IoU':>6}  {'Accuracy':>9}")
print(f"{'-'*50}")
for c in range(nc):
    class_mask = (inter[c] + (union[c]-inter[c]))
    class_acc  = inter[c] / (class_mask + 1e-8) * 100
    bar        = "#" * int(iou[c]*20)
    print(f" {CLASS_NAMES[c]:<22} {iou[c]:.3f}   {class_acc:>6.1f}%  {bar}")
print(f"{'-'*50}")
print(f" {'mIoU':<22} {miou:.3f}   {miou*100:>6.1f}%")
print(f"{'='*50}")

# -- Grade ---------------------------------------------------------
if miou >= 0.75:   grade = "[BEST] Excellent"
elif miou >= 0.60: grade = "[OK] Good"
elif miou >= 0.45: grade = "[!]  Acceptable"
else:              grade = "[FAIL] Needs improvement"
print(f"\n  Model Grade: {grade}")
print(f" Pixel Accuracy: {accuracy:.1f}% of all pixels correctly classified")

In [ ]:
# picks up from the epoch 30 checkpoint, trains to 50 total at a lower lr.
# redeclares everything so it works standalone after a restart

import numpy as np
import os, glob, random, time, warnings
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

WORK  = "/kaggle/working"
CACHE = None
for root,dirs,files in os.walk("/kaggle/input"):
    npy = [f for f in files if f.endswith("_bev.npy")]
    if len(npy) > 100:
        CACHE = root
        break
print(f"[OK] Cache : {CACHE}")

CFG = dict(
    cache_dir    = CACHE,
    checkpoint   = f"{WORK}/best_terrain_model.pt",
    history_file = f"{WORK}/training_history.npy",
    img_size     = 256,
    num_classes  = 5,
    batch_size   = 64,
    epochs       = 50,
    lr           = 1e-4,
    device       = "cuda" if torch.cuda.is_available() else "cpu",
)

CLASS_NAMES = {0:"Unknown",1:"Flat/Traversable",2:"Slope",3:"Obstacle",4:"Void"}
print(f"[OK] Device: {CFG['device']} | GPU: {torch.cuda.get_device_name(0)}")

# -- Dataset -------------------------------------------------------
class CachedTerrainDataset(Dataset):
    def __init__(self,pairs,cfg,augment=False):
        self.pairs=pairs; self.cache_dir=cfg["cache_dir"]
        self.augment=augment; self.sz=cfg["img_size"]
    def __len__(self): return len(self.pairs)
    def __getitem__(self,idx):
        key,_=self.pairs[idx]
        try:
            bev=np.load(os.path.join(self.cache_dir,f"{key}_bev.npy"))
            lbl=np.load(os.path.join(self.cache_dir,f"{key}_lbl.npy"))
            if self.augment: bev,lbl=self._augment(bev,lbl)
            return (torch.from_numpy(bev.transpose(2,0,1)).float(),
                    torch.from_numpy(lbl).long())
        except:
            sz=self.sz
            return (torch.zeros(5,sz,sz,dtype=torch.float32),
                    torch.zeros(sz,sz,dtype=torch.long))
    def _augment(self,bev,lbl):
        if random.random()>.5: bev,lbl=np.fliplr(bev).copy(),np.fliplr(lbl).copy()
        if random.random()>.5: bev,lbl=np.flipud(bev).copy(),np.flipud(lbl).copy()
        k=random.randint(0,3); bev,lbl=np.rot90(bev,k).copy(),np.rot90(lbl,k).copy()
        return bev,lbl

# -- Model ---------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self,ic,oc,drop=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(oc,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class UNet(nn.Module):
    def __init__(self,in_ch=5,nc=5,feats=[32,64,128,256]):
        super().__init__()
        self.downs=nn.ModuleList(); self.ups=nn.ModuleList(); self.pool=nn.MaxPool2d(2)
        ch=in_ch
        for f in feats: self.downs.append(DoubleConv(ch,f)); ch=f
        self.bot=DoubleConv(ch,ch*2); ch=ch*2
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(ch,f,2,stride=2))
            self.ups.append(DoubleConv(f*2,f)); ch=f
        self.head=nn.Conv2d(ch,nc,1); self._init()
    def _init(self):
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm2d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        skips=[]
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        x=self.bot(x); skips=skips[::-1]
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[i//2]
            if x.shape!=s.shape:
                x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=False)
            x=self.ups[i+1](torch.cat([s,x],1))
        return self.head(x)

# -- Pairs ---------------------------------------------------------
all_keys=[os.path.basename(f).replace("_bev.npy","")
          for f in glob.glob(os.path.join(CACHE,"*_bev.npy"))]
random.seed(42); random.shuffle(all_keys)
split  = int(len(all_keys)*0.8)
tr_p   = [(k,k) for k in all_keys[:split]]
va_p   = [(k,k) for k in all_keys[split:]]
print(f"[OK] Train: {len(tr_p):,} | Val: {len(va_p):,}")

# -- DataLoaders ---------------------------------------------------
tr_dl = DataLoader(CachedTerrainDataset(tr_p,CFG,augment=True),
                   batch_size=CFG["batch_size"],shuffle=True,
                   num_workers=2,pin_memory=True)
va_dl = DataLoader(CachedTerrainDataset(va_p,CFG,augment=False),
                   batch_size=CFG["batch_size"],shuffle=False,
                   num_workers=2,pin_memory=True)

# -- Load checkpoint -----------------------------------------------
model=UNet(in_ch=5,nc=5).to(CFG["device"])
model.load_state_dict(torch.load(CFG["checkpoint"],
                                  map_location=CFG["device"]))
print(f"[OK] Loaded checkpoint  --  resuming from epoch 30")

# -- Optimizer -----------------------------------------------------
w      = torch.tensor([0.5,1.0,1.5,2.5,2.0]).to(CFG["device"])
crit   = nn.CrossEntropyLoss(weight=w)
opt    = torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=20,eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

# -- Load history --------------------------------------------------
if os.path.exists(CFG["history_file"]):
    hist=np.load(CFG["history_file"],allow_pickle=True).item()
    print(f"[OK] History: {len(hist['train'])} epochs | best: {min(hist['val']):.4f}")
else:
    hist={"train":[],"val":[]}

best        = min(hist["val"]) if hist["val"] else float("inf")
start_epoch = len(hist["train"])+1
print(f"[OK] Resuming epoch {start_epoch} -> {CFG['epochs']}\n")
print(f"{'Ep':>4} {'Train':>8} {'Val':>8} {'LR':>9} {'GPU%':>5} {'Min':>5}")
print("-"*42)

# -- Train ---------------------------------------------------------
for ep in range(start_epoch,CFG["epochs"]+1):
    model.train(); tl=0.0; t0=time.time()
    for bev,lbl in tr_dl:
        bev=bev.to(CFG["device"]); lbl=lbl.to(CFG["device"])
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss=crit(model(bev),lbl)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt); scaler.update()
        tl+=loss.item()
    tl/=len(tr_dl); sched.step()

    model.eval(); vl=0.0
    with torch.no_grad():
        for bev,lbl in va_dl:
            bev=bev.to(CFG["device"]); lbl=lbl.to(CFG["device"])
            with torch.cuda.amp.autocast():
                vl+=crit(model(bev),lbl).item()
    vl/=len(va_dl)

    hist["train"].append(tl); hist["val"].append(vl)
    star=""
    if vl<best:
        best=vl
        torch.save(model.state_dict(),CFG["checkpoint"])
        np.save(CFG["history_file"],hist)
        star=" *"

    try: gpu=torch.cuda.utilization()
    except: gpu=0
    ep_t=time.time()-t0
    print(f"{ep:>4}  {tl:>8.4f}  {vl:>8.4f}  "
          f"{sched.get_last_lr()[0]:>8.1e}  "
          f"{gpu:>4}%  {ep_t/60:>4.1f}m{star}")

print(f"\n[OK] Done! Best val: {best:.4f}")

# -- Plot ----------------------------------------------------------
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(hist["train"],lw=2,label="Train")
ax.plot(hist["val"],  lw=2,label="Val")
ax.axvline(29,color="grey",lw=1.5,linestyle="--",label="Previous stop (ep 30)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training History  --  50 Epochs")
ax.legend(); ax.grid(alpha=.4)
plt.tight_layout()
plt.savefig(f"{WORK}/training_curve_50ep.png",dpi=150,bbox_inches="tight")
plt.show()

# -- Updated accuracy ----------------------------------------------
print("\n=== Updated Accuracy ===")
nc=5
inter=np.zeros(nc,np.float64); union=np.zeros(nc,np.float64)
tp_all=0; total=0
model.eval()
with torch.no_grad():
    for bev,lbl in DataLoader(CachedTerrainDataset(va_p,CFG,augment=False),
                               batch_size=64,num_workers=2):
        bev=bev.to(CFG["device"])
        with torch.cuda.amp.autocast():
            pred=model(bev).argmax(1).cpu().numpy().flatten()
        true=lbl.numpy().flatten()
        tp_all+=(pred==true).sum(); total+=len(true)
        for c in range(nc):
            inter[c]+=((pred==c)&(true==c)).sum()
            union[c]+=((pred==c)|(true==c)).sum()

iou=inter/(union+1e-8); miou=iou.mean(); acc=tp_all/total*100
print(f"\n  Pixel Accuracy : {acc:.2f}%  (was 98.63%)")
print(f" mIoU           : {miou:.3f}  (was 0.599)")
print(f"{'-'*45}")
for c in range(nc):
    print(f" {CLASS_NAMES[c]:<22} {iou[c]:.3f}  {'#'*int(iou[c]*20)}")
print(f"{'-'*45}")
print(f" mIoU : {miou:.3f}")
if miou>=0.75:   grade="[BEST] Excellent"
elif miou>=0.60: grade="[OK] Good"
elif miou>=0.45: grade="[!]  Acceptable"
else:            grade="[FAIL] Needs improvement"
print(f" Grade: {grade}")

In [ ]:
# same eval report as before, rerun on the model after the extra 20 epochs

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import numpy as np
import torch

class CachedTerrainDataset(torch.utils.data.Dataset):
    def __init__(self, pairs, cfg):
        self.pairs=pairs; self.cache_dir=cfg["cache_dir"]; self.sz=cfg["img_size"]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        key,_=self.pairs[idx]
        try:
            bev=np.load(os.path.join(self.cache_dir,f"{key}_bev.npy"))
            lbl=np.load(os.path.join(self.cache_dir,f"{key}_lbl.npy"))
            return (torch.from_numpy(bev.transpose(2,0,1)).float(),
                    torch.from_numpy(lbl).long())
        except:
            sz=self.sz
            return (torch.zeros(5,sz,sz), torch.zeros(sz,sz,dtype=torch.long))

# -- Build val pairs -----------------------------------------------
all_keys = [os.path.basename(f).replace("_bev.npy","")
            for f in glob.glob(os.path.join(CACHE,"*_bev.npy"))]
random.seed(42); random.shuffle(all_keys)
split    = int(len(all_keys)*0.8)
va_keys  = all_keys[split:]
va_pairs = [(k,k) for k in va_keys]
print(f"Evaluating on {len(va_pairs):,} validation frames...")

dl = DataLoader(CachedTerrainDataset(va_pairs, CFG),
                batch_size=64, shuffle=False, num_workers=2)

# -- Metrics -------------------------------------------------------
nc     = CFG["num_classes"]
inter  = np.zeros(nc, np.float64)
union  = np.zeros(nc, np.float64)
tp_all = 0
total  = 0

model.eval()
with torch.no_grad():
    for bev, lbl in tqdm(dl, desc="Evaluating"):
        bev  = bev.to(CFG["device"])
        with torch.cuda.amp.autocast():
            pred = model(bev).argmax(1).cpu().numpy().flatten()
        true = lbl.numpy().flatten()

        # Pixel accuracy
        tp_all += (pred == true).sum()
        total  += len(true)

        # IoU per class
        for c in range(nc):
            inter[c] += ((pred==c) & (true==c)).sum()
            union[c] += ((pred==c) | (true==c)).sum()

iou      = inter / (union + 1e-8)
miou     = iou.mean()
accuracy = tp_all / total * 100

# -- Results -------------------------------------------------------
print(f"\n{'='*50}")
print(f" MODEL ACCURACY REPORT")
print(f"{'='*50}")
print(f" Overall Pixel Accuracy : {accuracy:.2f}%")
print(f" Mean IoU (mIoU)        : {miou:.3f} ({miou*100:.1f}%)")
print(f"{'-'*50}")
print(f" {'Class':<22} {'IoU':>6}  {'Accuracy':>9}")
print(f"{'-'*50}")
for c in range(nc):
    class_mask = (inter[c] + (union[c]-inter[c]))
    class_acc  = inter[c] / (class_mask + 1e-8) * 100
    bar        = "#" * int(iou[c]*20)
    print(f" {CLASS_NAMES[c]:<22} {iou[c]:.3f}   {class_acc:>6.1f}%  {bar}")
print(f"{'-'*50}")
print(f" {'mIoU':<22} {miou:.3f}   {miou*100:>6.1f}%")
print(f"{'='*50}")

# -- Grade ---------------------------------------------------------
if miou >= 0.75:   grade = "[BEST] Excellent"
elif miou >= 0.60: grade = "[OK] Good"
elif miou >= 0.45: grade = "[!]  Acceptable"
else:              grade = "[FAIL] Needs improvement"
print(f"\n  Model Grade: {grade}")
print(f" Pixel Accuracy: {accuracy:.1f}% of all pixels correctly classified")